# Unit 10 — キャップストーン: アンサンブルから推論運用まで

目安は **20〜25分**。ここまで作ってきた「リークのない検証 → 改善 → 予測」を、実務で繰り返せる1本の流れに束ねます。

今日のゴールは次の5つです。

1. OOF 予測を重み付き平均・順位平均し、test を見ずに重みを決める
2. stacking、seed 平均、予測相関の役割を区別する
3. 学習成果物を artifact として保存し、推論では再学習しない
4. GPU学習 + CPU常時推論と LLM API の費用を一般式で比較する
5. 精度・レイテンシ・メモリ・月額を満たす end-to-end 構成を選ぶ

外部 API やクラウドには接続しません。価格はすべて関数の引数にし、特定サービスの現行単価には依存しない教材です。

> C# なら、学習は `dotnet build`、artifact はビルド成果物、推論はデプロイ済みバイナリの実行です。運用中に毎回ビルドし直さないのと同じ分離を行います。

In [ ]:
# STEP 1: Unit 10 — キャップストーン: アンサンブルから推論運用までの処理を実行し、出力を照合する
from pathlib import Path
import tempfile
import platform

import joblib
import numpy as np
import pandas as pd
import sklearn
from scipy.stats import rankdata
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

UNIT_DIR = Path(".")
if not (UNIT_DIR / "lesson.ipynb").exists():
    UNIT_DIR = Path("courses/kaggle-sprint/unit10-capstone-ensemble-and-serving")
if not (UNIT_DIR / "lesson.ipynb").exists():
    raise FileNotFoundError("unit10 またはリポジトリ直下から実行してください。")

print("unit:", UNIT_DIR.resolve())
print("numpy:", np.__version__, "/ sklearn:", sklearn.__version__)

def check(name, actual, expected, hint=""):
    import numpy as _np
    try:
        ok = actual is not None and bool(_np.all(_np.isclose(
            _np.asarray(actual, dtype=float), _np.asarray(expected, dtype=float)
        )))
    except (TypeError, ValueError):
        ok = actual == expected
    mark = "OK" if ok else "NG"
    if ok:
        print(f"[OK] {name}")
    else:
        detail = f"actual={actual!r} / expected={expected!r}"
        print(f"[NG] {name} — {detail}" + ("" if not hint else f" — ヒント: {hint}"))
    return ok

def safe_call(fn):
    try:
        return fn()
    except Exception:
        return None

def safe_get(mapping, key):
    return mapping.get(key) if isinstance(mapping, dict) else None

## 今日のパイプライン

**学習側**: split を固定 → base model の OOF を作る → blend / meta model を決める → 前処理・列順・閾値・seed・version と一緒に保存

**推論側**: artifact を読む → schema / drift を検査 → 同じ前処理 → base 予測 → ensemble → 出力検査 → 監視値を記録

各ブロックを **見る → 予測 → 変える → 書く → チェック** の順で進めます。shape と axis が出たら、まず `print` で確認します。

## 1. OOF 予測だけで blend 重みを決める

OOF (out-of-fold) 予測は、各行を「その行を学習に使っていないモデル」で予測した値です。モデルごとの OOF を横に並べると shape は `(n_train, n_models)`。重み `w` との積 `OOF @ w` で blend できます。

`scipy.stats.rankdata` は値を順位へ変換する API です。rank average はモデルごとの尺度差を消せるため順位系指標で有効なことがありますが、確率校正は失います。

重要なのは、重み探索も meta model 学習も **OOF 上だけ**で行い、test や leaderboard を見て決めないことです。

In [ ]:
# STEP 2: 1. OOF 予測だけで blend 重みを決めるの処理を実行し、出力を照合する
# 学習済み3モデルから得た genuine OOF を模した、固定の小データ。
rng = np.random.default_rng(10)
n_samples = 120
signals = rng.normal(size=(n_samples, 3))
y_oof = (0.9 * signals[:, 0] - 0.5 * signals[:, 1] + rng.normal(0, 0.7, n_samples) > 0).astype(int)

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

oof_matrix = np.column_stack([
    sigmoid(1.0 * signals[:, 0] - 0.2 * signals[:, 1] + rng.normal(0, 0.5, n_samples)),
    sigmoid(0.3 * signals[:, 0] - 0.9 * signals[:, 1] + rng.normal(0, 0.6, n_samples)),
    sigmoid(0.7 * signals[:, 0] - 0.5 * signals[:, 1] + rng.normal(0, 0.8, n_samples)),
])
print("OOF shape:", oof_matrix.shape, "/ target:", y_oof.shape)

candidates = []
for w0 in np.linspace(0, 1, 11):
    for w1 in np.linspace(0, 1 - w0, 11):
        weights = np.array([w0, w1, 1 - w0 - w1])
        prediction = oof_matrix @ weights
        candidates.append((brier_score_loss(y_oof, prediction), weights))
best_loss, best_weights = min(candidates, key=lambda item: item[0])
print("best weights:", best_weights.round(2), "/ Brier:", round(best_loss, 4))

In [ ]:
# STEP 3: 1. OOF 予測だけで blend 重みを決めるの処理を実行し、出力を照合する
equal_prediction = oof_matrix.mean(axis=1)
rank_matrix = np.column_stack([
    rankdata(oof_matrix[:, column], method="average") / (n_samples + 1)
    for column in range(oof_matrix.shape[1])
])
rank_prediction = rank_matrix.mean(axis=1)

print("equal Brier:", round(brier_score_loss(y_oof, equal_prediction), 4))
print("weighted Brier:", round(brier_score_loss(y_oof, oof_matrix @ best_weights), 4))
print("rank shape/range:", rank_matrix.shape, (rank_prediction.min().round(3), rank_prediction.max().round(3)))

### 予測

OOF shape が `(120, 3)`、重み shape が `(3,)` のとき、`OOF @ weights` の shape はどれでしょう。

A. `(120,)`　B. `(3,)`　C. `(120, 3)`

各行について3モデルを1つの予測へ畳み込みます。

In [ ]:
# STEP 4: 予測の処理を実行し、出力を照合する
prediction_1 = "A"
print("答え:", prediction_1, "— C# の各行 Select(row => Dot(row, weights)) に相当します")

### 変える → 書く

`best_weights` を単純平均 `[1/3, 1/3, 1/3]` に替え、損失の変化を見てください。探索刻みを細かくすれば候補は増えますが、OOF 自体への過学習も強くなります。

次に `weighted_blend` を完成させます。入力は `(n_rows, n_models)`、重みは `(n_models,)`。重み和が1であることも検査してください。

In [ ]:
# STEP 5: 変える → 書くの処理を実行し、出力を照合する
def weighted_blend(prediction_matrix, weights):
    # TODO: 配列化 → shape/重み和を検査 → 行列積を返す
    return None

blend_input = np.array([[0.1, 0.2, 0.4], [0.8, 0.6, 0.7], [0.3, 0.9, 0.6]])
blend_weights = np.array([0.5, 0.3, 0.2])
learner_blend = safe_call(lambda: weighted_blend(blend_input, blend_weights))

In [ ]:
# STEP 6: 変える → 書くの処理を実行し、出力を照合する
check("blend shape", safe_call(lambda: learner_blend.shape), (3,), "モデル軸が消えます")
check("blend values", learner_blend, [0.19, 0.72, 0.54], "各行と weights の内積です")
check("blend dtype", safe_call(lambda: str(learner_blend.dtype)), "float64", "dtype=float で配列化します")
check("blend probability range", safe_call(lambda: bool(np.all((learner_blend >= 0) & (learner_blend <= 1)))), True, "確率の凸結合は範囲内です")

## 2. stacking・seed平均・予測相関を使い分ける

stacking は OOF 列を新しい特徴量として、弱い meta model を学習します。`cross_val_predict` は各 fold の未学習行への予測を元の行順で返す API です。meta model の性能を見るときも、さらに CV 内で予測しないと同じ OOF を学習・評価してしまいます。

seed 平均は同じ構成の乱数だけを変え、予測を平均して分散を減らします。異種モデルを混ぜるときは `np.corrcoef(..., rowvar=False)` で予測相関を見ます。単体精度が同程度なら、誤りが異なる低相関モデルほど補完しやすい傾向があります。

In [ ]:
# STEP 7: 2. stacking・seed平均・予測相関を使い分けるの処理を実行し、出力を照合する
meta_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=10)
meta_model = LogisticRegression(C=0.5, max_iter=1000, random_state=10)
meta_oof = cross_val_predict(
    meta_model, oof_matrix, y_oof,
    cv=meta_cv, method="predict_proba"
)[:, 1]
meta_loss = brier_score_loss(y_oof, meta_oof)

correlation = np.corrcoef(oof_matrix, rowvar=False)
print("meta OOF shape:", meta_oof.shape, "/ Brier:", round(meta_loss, 4))
print("base prediction correlation:\n", np.round(correlation, 3))

In [ ]:
# STEP 8: 2. stacking・seed平均・予測相関を使い分けるの処理を実行し、出力を照合する
seed_predictions = np.array([
    [0.20, 0.80, 0.40, 0.60],
    [0.30, 0.70, 0.50, 0.50],
    [0.25, 0.75, 0.45, 0.55],
])
seed_mean = seed_predictions.mean(axis=0)
seed_correlation = np.corrcoef(seed_predictions)
print("seed tensor:", seed_predictions.shape, "= (seeds, rows)")
print("mean prediction:", seed_mean)
print("seed correlation:\n", np.round(seed_correlation, 3))

### 予測

seed 平均を3個から10個へ増やせば、必ず大きく改善するでしょうか。

A. 必ず改善する　B. 改善幅は逓減し、学習費との比較が必要

同じ構成の高相関な予測を増やすほど、新しく消せる誤差は小さくなります。

In [ ]:
# STEP 9: 予測の処理を実行し、出力を照合する
prediction_2 = "B"
print("答え:", prediction_2, "— seed数はCV改善量 / 追加学習時間で決めます")

### 変える → 書く

`seed_predictions` の3本目を1本目と完全に同じにし、相関と平均を見てください。予測本数が増えても情報の種類は増えません。

`ensemble_summary` を完成させ、seed 軸を平均した予測と seed 間相関を辞書で返してください。`np.corrcoef` は既定で各行を変数とみなすので、この入力では `rowvar` を替える必要はありません。

In [ ]:
# STEP 10: 変える → 書くの処理を実行し、出力を照合する
def ensemble_summary(predictions_by_seed):
    # TODO: float配列化し、axis=0平均とseed間相関を返す
    return None

learner_seed_summary = safe_call(lambda: ensemble_summary(seed_predictions))

In [ ]:
# STEP 11: 変える → 書くの処理を実行し、出力を照合する
check("seed mean", safe_get(learner_seed_summary, "mean"), [0.25, 0.75, 0.45, 0.55], "seed 軸 axis=0 を平均します")
check("seed correlation shape", safe_call(lambda: safe_get(learner_seed_summary, "correlation").shape), (3, 3), "seed数×seed数です")
check("seed correlation diagonal", safe_call(lambda: np.diag(safe_get(learner_seed_summary, "correlation"))), [1, 1, 1], "自分自身との相関は1です")
check("seed correlation pair", safe_call(lambda: safe_get(learner_seed_summary, "correlation")[0, 1]), 0.9486832981, "1本目と2本目の相関です")

## 3. 学習と推論を artifact で分離する

学習側が保存すべきものはモデル本体だけではありません。

| artifact | なぜ必要か |
|---|---|
| fit済み前処理器 + model | 推論時に同じ変換・重みを使う |
| feature_order / dtype | 列入替や上流 schema 変更を検知する |
| threshold / class mapping | 確率を業務出力へ変換する |
| seed / library version | 再現・障害調査を可能にする |
| 学習時の分布要約 | 欠損率・平均・カテゴリ比の drift と比較する |

`joblib.dump` / `joblib.load` は sklearn オブジェクトを保存・復元する API です。信頼できないファイルは読み込まず、環境のライブラリ version も固定します。

In [ ]:
# STEP 12: 3. 学習と推論を artifact で分離するの処理を実行し、出力を照合する
rng_artifact = np.random.default_rng(30)
feature_order = ["views", "description_length", "quality_score"]
training_features = pd.DataFrame({
    "views": rng_artifact.normal(100, 25, 80),
    "description_length": rng_artifact.integers(20, 300, 80),
    "quality_score": rng_artifact.uniform(0, 1, 80),
})
training_target = (
    0.018 * training_features["views"]
    + 1.6 * training_features["quality_score"]
    - 0.003 * training_features["description_length"]
    + rng_artifact.normal(0, 0.6, 80)
    > 1.7
).astype(int)

pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=30),
)
pipeline.fit(training_features[feature_order], training_target)

artifact_bundle = {
    "pipeline": pipeline,
    "feature_order": feature_order,
    "threshold": 0.55,
    "metadata": {
        "seed": 30,
        "python": platform.python_version(),
        "numpy": np.__version__,
        "sklearn": sklearn.__version__,
    },
    "baseline_missing_rate": training_features.isna().mean().to_dict(),
    "baseline_mean": training_features.mean().to_dict(),
}

with tempfile.TemporaryDirectory() as temporary_directory:
    artifact_path = Path(temporary_directory) / "model_bundle.joblib"
    joblib.dump(artifact_bundle, artifact_path)
    loaded_artifact = joblib.load(artifact_path)
    print("saved bytes:", artifact_path.stat().st_size)
print("loaded keys:", sorted(loaded_artifact))

In [ ]:
# STEP 13: 3. 学習と推論を artifact で分離するの処理を実行し、出力を照合する
serving_frame = training_features.sample(5, random_state=2)[
    ["quality_score", "views", "description_length"]
]
ordered_serving = serving_frame.loc[:, loaded_artifact["feature_order"]]
serving_probability = loaded_artifact["pipeline"].predict_proba(ordered_serving)[:, 1]
serving_label = (serving_probability >= loaded_artifact["threshold"]).astype(int)
print("incoming columns:", serving_frame.columns.tolist())
print("model columns:", ordered_serving.columns.tolist())
print("probability/label:", np.round(serving_probability, 3), serving_label)

drift_frame = serving_frame.copy()
drift_frame.loc[drift_frame.index[0], "views"] = np.nan
current_missing = drift_frame.isna().mean()
baseline_missing = pd.Series(loaded_artifact["baseline_missing_rate"])
print("missing-rate delta:\n", (current_missing - baseline_missing).round(3))

### 予測

入力列が同じ3列でも順番だけ違う場合、そのまま NumPy 配列にして推論してよいでしょうか。

A. よい　B. artifact の feature_order に並べ直す

sklearn は DataFrame 名を検査することがありますが、配列へ落とした後は列名がありません。推論入口で明示的に揃えます。

In [ ]:
# STEP 14: 予測の処理を実行し、出力を照合する
prediction_3 = "B"
print("答え:", prediction_3, "— schema は推論 API の型契約です")

### 変える → 書く

`serving_frame` から1列を削除し、推論前に止めるべき状態を確認してください。欠損列を0で黙って補うと、上流障害を正常入力として処理してしまいます。

`align_schema` を完成させます。missing / extra を列挙し、完全一致なら artifact の順番へ並べた NumPy 配列も返してください。これは unit01 の健全性チェックの本番版です。

In [ ]:
# STEP 15: 変える → 書くの処理を実行し、出力を照合する
def align_schema(frame, expected_columns):
    # TODO: missing/extra を求め、完全一致なら expected 順の values を返す
    return None

schema_frame = pd.DataFrame({"b": [2, 4], "a": [1, 3]})
valid_schema = safe_call(lambda: align_schema(schema_frame, ["a", "b"]))
missing_schema = safe_call(lambda: align_schema(schema_frame[["a"]], ["a", "b"]))

In [ ]:
# STEP 16: 変える → 書くの処理を実行し、出力を照合する
check("valid schema", safe_get(valid_schema, "ok"), True, "missing と extra が空なら True")
check("aligned first row", safe_call(lambda: safe_get(valid_schema, "values")[0].tolist()), [1, 2], "expected_columns 順に .loc します")
check("missing column", safe_get(missing_schema, "missing"), ["b"], "期待列にあって入力にない列です")
check("invalid schema", safe_get(missing_schema, "ok"), False, "missing があれば False")

## 4. GPU学習 + CPU推論と API の損益分岐を式にする

現行価格を暗記せず、見積り時点の値を引数で入れます。

**API月額** = 月間件数 × (入力token + 出力token) ÷ 1,000,000 × token単価

**自前月額** = GPU時間単価 × 学習時間 × 月間再学習回数 + CPU時間単価 × 月間稼働時間

常時CPUではなくバッチ推論なら、推論項を `インスタンス時間単価 × (件数 / throughput) / 3600` に替えます。storage・通信・監視・人件費も本番見積りでは別項目として加えます。

In [ ]:
# STEP 17: 4. GPU学習 + CPU推論と API の損益分岐を式にするの処理を実行し、出力を照合する
def reference_monthly_cost(config):
    requests = config["daily_requests"] * config["days"]
    tokens_per_request = config["input_tokens"] + config["output_tokens"]
    api = requests * tokens_per_request / 1_000_000 * config["api_price_per_million"]
    training = (
        config["gpu_hour_rate"] * config["training_hours"] * config["retrains_per_month"]
    )
    inference = config["cpu_hour_rate"] * config["cpu_hours_per_month"]
    self_hosted = training + inference
    api_per_request = tokens_per_request / 1_000_000 * config["api_price_per_million"]
    break_even = self_hosted / api_per_request if api_per_request > 0 else np.inf
    return {"api": api, "self_hosted": self_hosted, "break_even_requests": break_even}

# 単価は式を動かすための架空シナリオであり、特定サービスの価格ではありません。
cost_scenario = {
    "daily_requests": 5_000,
    "days": 30,
    "input_tokens": 700,
    "output_tokens": 100,
    "api_price_per_million": 2.5,
    "gpu_hour_rate": 4.0,
    "training_hours": 2.5,
    "retrains_per_month": 2,
    "cpu_hour_rate": 0.12,
    "cpu_hours_per_month": 24 * 30,
}
reference_cost = reference_monthly_cost(cost_scenario)
reference_cost

In [ ]:
# STEP 18: 4. GPU学習 + CPU推論と API の損益分岐を式にするの処理を実行し、出力を照合する
for key, value in reference_cost.items():
    print(f"{key}: {value:,.2f}")
print("daily break-even:", round(reference_cost["break_even_requests"] / cost_scenario["days"]))

print("\n感度分析: 件数だけ変更")
for daily in [500, 2_000, 5_000, 20_000]:
    changed = dict(cost_scenario, daily_requests=daily)
    result = reference_monthly_cost(changed)
    cheaper = "self-hosted" if result["self_hosted"] < result["api"] else "API"
    print(f"{daily:>6,}/day API={result['api']:>7.1f} self={result['self_hosted']:>7.1f} -> {cheaper}")

### 予測

件数が2倍になったとき、CPUを常時稼働する自前月額と API 月額はどう変わりますか。

A. 両方2倍　B. 自前のこの式は一定、APIは2倍　C. 自前だけ2倍

実際には負荷が容量を超えれば CPU 台数を増やす段差があります。まず1台の容量内の式を読んでください。

In [ ]:
# STEP 19: 予測の処理を実行し、出力を照合する
prediction_4 = "B"
print("答え:", prediction_4, "— 固定費と従量費の交点が損益分岐です")

### 変える → 書く

単価・token数・再学習回数・CPU稼働時間を1つずつ変え、どれが意思決定を反転させるか見てください。損益分岐を1点で断言せず、低位・基準・高位の3シナリオにすると予算説明が強くなります。

`estimate_month` を完成させ、API月額、自前月額、月間損益分岐件数を返してください。価格はすべて `config` から受け取ります。

In [ ]:
# STEP 20: 変える → 書くの処理を実行し、出力を照合する
def estimate_month(config):
    # TODO: API従量費、自前のGPU学習+CPU常時費、損益分岐件数を計算
    return None

learner_cost = safe_call(lambda: estimate_month(cost_scenario))

In [ ]:
# STEP 21: 変える → 書くの処理を実行し、出力を照合する
check("API monthly cost", safe_get(learner_cost, "api"), 300.0, "件数×token×100万token単価")
check("self-hosted monthly cost", safe_get(learner_cost, "self_hosted"), 106.4, "GPU学習20 + CPU常時86.4")
check("break-even requests", safe_get(learner_cost, "break_even_requests"), 53200.0, "自前月額 / 1件API費")
check("cheaper at scenario volume", safe_call(lambda: safe_get(learner_cost, "self_hosted") < safe_get(learner_cost, "api")), True, "15万件/月で比較します")

## 5. trade-off を測り、end-to-end を閉じる

推論高速化には必ず交換条件があります。

| 手段 | 主な利点 | 主な注意 |
|---|---|---|
| batch拡大 | throughput↑ | 単件待ち時間・memory↑ |
| dynamic padding | 無駄token↓ | 長さ別batch設計が必要 |
| `inference_mode` / `no_grad` | 勾配memory・計算を削減 | `model.eval()` は別途必要 |
| 系列長上限 | コスト↓ | 情報切捨てで精度↓ |
| 蒸留・小型model | latency / memory↓ | 精度検証が必要 |
| 量子化・混合精度 | memory・演算量↓ | hardware対応・数値誤差 |
| cache | 同一入力を再計算しない | 無効化・個人情報管理 |

GPU学習を spot で行うなら checkpoint から再開可能にします。GPUで学習した PyTorch 重みを CPU へ運ぶときは `torch.load(..., map_location="cpu")`、推論時は `model.eval()` と `torch.inference_mode()` を使います。

In [ ]:
# STEP 22: 5. trade-off を測り、end-to-end を閉じるの処理を実行し、出力を照合する
lengths = np.array([5, 8, 3, 4])
static_max_length = 12
dynamic_tokens = len(lengths) * lengths.max()
static_tokens = len(lengths) * static_max_length
print("actual tokens:", lengths.sum())
print("dynamic padding:", dynamic_tokens)
print("static padding:", static_tokens)
print("dynamic saving:", f"{1 - dynamic_tokens/static_tokens:.1%}")

serving_configs = [
    {"name": "fp32_cpu_b1", "accuracy": 0.910, "p95_ms": 35, "memory_mb": 900, "monthly_cost": 120},
    {"name": "int8_cpu_b16", "accuracy": 0.905, "p95_ms": 80, "memory_mb": 350, "monthly_cost": 70},
    {"name": "fp16_gpu_b64", "accuracy": 0.912, "p95_ms": 45, "memory_mb": 700, "monthly_cost": 260},
]
pd.DataFrame(serving_configs)

In [ ]:
# STEP 23: 5. trade-off を測り、end-to-end を閉じるの処理を実行し、出力を照合する
def run_end_to_end(frame, artifact):
    expected = artifact["feature_order"]
    missing = [column for column in expected if column not in frame.columns]
    extra = [column for column in frame.columns if column not in expected]
    if missing or extra:
        raise ValueError(f"schema mismatch: missing={missing}, extra={extra}")
    ordered = frame.loc[:, expected]
    probability = artifact["pipeline"].predict_proba(ordered)[:, 1]
    return pd.DataFrame({
        "row_id": np.arange(len(frame)),
        "prediction": probability,
        "label": (probability >= artifact["threshold"]).astype(int),
    })

submission_1 = run_end_to_end(serving_frame, loaded_artifact)
submission_2 = run_end_to_end(serving_frame, loaded_artifact)
print(submission_1)
print("same input -> same output:", np.allclose(submission_1["prediction"], submission_2["prediction"]))
print("output finite:", np.isfinite(submission_1["prediction"]).all())

### 予測

精度制約 `>= 0.90`、p95 latency 制約 `<= 100ms` のもとで、表の最小月額構成はどれでしょう。

A. `fp32_cpu_b1`　B. `int8_cpu_b16`　C. `fp16_gpu_b64`

「最速」ではなく「制約を満たす中で最小コスト」を選びます。

In [ ]:
# STEP 24: 予測の処理を実行し、出力を照合する
prediction_5 = "B"
print("答え:", prediction_5, "— int8構成は精度・latency制約を満たし、月額70です")

### 変える → 書く

最小精度を `0.91`、最大p95を `50ms` に変え、選択がどう変わるか確認してください。制約を満たす構成がなければ、黙って最も近いものを返さず `None` にします。

`padded_tokens` は1 batch の dynamic padding token数を返し、`choose_config` は制約内の最小月額構成を返します。動的 padding は batch 内最大長まで揃えるため `len(lengths) * max(lengths)` です。

In [ ]:
# STEP 25: 変える → 書くの処理を実行し、出力を照合する
def padded_tokens(sequence_lengths):
    # TODO: batch件数 × batch内最大長
    return None

def choose_config(configs, minimum_accuracy, maximum_p95_ms):
    # TODO: 制約を満たす構成を絞り、monthly_cost 最小を返す。なければ None
    return None

learner_dynamic_tokens = safe_call(lambda: padded_tokens(lengths))
learner_best_config = safe_call(lambda: choose_config(serving_configs, 0.90, 100))

In [ ]:
# STEP 26: 変える → 書くの処理を実行し、出力を照合する
check("dynamic padded tokens", learner_dynamic_tokens, 32, "4件×batch内最大長8")
check("padding saving", safe_call(lambda: 1 - learner_dynamic_tokens / static_tokens), 1 / 3, "1 - 32/48")
check("selected config", safe_get(learner_best_config, "name"), "int8_cpu_b16", "制約内で monthly_cost 最小")
check("selected monthly cost", safe_get(learner_best_config, "monthly_cost"), 70, "選んだ構成の月額です")

## 振り返り

次を自分の言葉で説明してください。

1. blend 重みや meta model を test 結果で決めると、なぜリークになりますか。
2. 高精度でも高相関な2モデルより、少し弱くても低相関なモデルが役立つ場合があるのはなぜですか。
3. artifact に列順と library version を含める理由は何ですか。
4. 自前推論と API の損益分岐を動かす主要な引数は何ですか。
5. batch、dynamic padding、量子化が変える精度・latency・throughput・memory を整理してください。

## まとめ — 1本の運用可能な流れ

1. split と seed を固定し、各 base model の OOF / test 予測を作る
2. OOF だけで blend・rank average・stacking を比較する
3. seed平均と予測相関を見て、追加学習の費用対効果を決める
4. model・前処理・列順・閾値・version・baseline分布を artifact 化する
5. 推論入口で schema / drift、出口で件数・有限値・再現性を検査する
6. API従量費と GPU学習 + CPU推論費を、最新の入力値で再計算する
7. 精度・p95・memory 制約内の最小コスト構成を選び、継続監視する

全チェックが `OK` になったら、ex01〜ex04 で OOF 重み探索から artifact 推論、コスト制約付きキャップストーンまで自力実装します。